# GHINOVA — Lesson 3 Lab: How Large Language Models Work
Hands-on companion for **Module 1 · Lesson 3**. No GPU or API key required.

You will experiment with tokenization, token IDs, logits, softmax, temperature, sampling, and a tiny next-token language model.

## 1. A simple tokenizer
Real LLMs use learned tokenizers. This small regex tokenizer makes the idea easy to inspect.

In [ ]:
import re, math, random
from collections import Counter, defaultdict

def simple_tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

text = "Large language models predict the next token!"
tokens = simple_tokenize(text)
print(tokens)
print('Number of tokens:', len(tokens))

## 2. Tokens → IDs
A vocabulary maps each token to an integer ID.

In [ ]:
vocab = {token: i for i, token in enumerate(sorted(set(tokens)))}
token_ids = [vocab[t] for t in tokens]
print('Vocabulary:', vocab)
print('Token IDs:', token_ids)

## 3. Logits → softmax → probabilities
An LLM produces scores called **logits**. Softmax converts them into a probability distribution.

In [ ]:
candidate_tokens = ['France', 'Europe', 'Paris', 'city']
logits = [3.0, 1.0, 0.6, 1.4]

def softmax(values, temperature=1.0):
    scaled = [v / temperature for v in values]
    m = max(scaled)
    exps = [math.exp(v-m) for v in scaled]
    total = sum(exps)
    return [e/total for e in exps]

probs = softmax(logits)
for token, p in zip(candidate_tokens, probs):
    print(f'{token:8s}: {p:.3f}')

## 4. Temperature
Lower temperature sharpens the distribution; higher temperature makes it flatter. Temperature changes sampling behavior — not the model's knowledge.

In [ ]:
for temp in [0.3, 1.0, 2.0]:
    probs = softmax(logits, temperature=temp)
    print(f'\nTemperature = {temp}')
    for token, p in zip(candidate_tokens, probs):
        print(f'{token:8s}: {p:.3f}')

## 5. Sample the next token

In [ ]:
def sample_next(logits, temperature=1.0):
    probs = softmax(logits, temperature)
    return random.choices(candidate_tokens, weights=probs, k=1)[0]

for temp in [0.3, 1.0, 2.0]:
    samples = [sample_next(logits, temp) for _ in range(20)]
    print(temp, Counter(samples))

## 6. Build a tiny next-token language model
This bigram model is far simpler than a Transformer, but it demonstrates the central idea: estimate the next token from previous context.

In [ ]:
corpus = '''
artificial intelligence can analyze data
artificial intelligence can generate text
machine learning can learn patterns
language models can generate text
language models predict the next token
machine learning uses data
'''
corpus_tokens = simple_tokenize(corpus)
counts = defaultdict(Counter)
for a, b in zip(corpus_tokens, corpus_tokens[1:]):
    counts[a][b] += 1

def next_token_distribution(token):
    c = counts[token]
    total = sum(c.values())
    return {t: n/total for t,n in c.items()} if total else {}

print("After 'can':", next_token_distribution('can'))
print("After 'language':", next_token_distribution('language'))

## 7. Generate with the tiny model

In [ ]:
def generate(start, steps=8):
    current = start.lower()
    output = [current]
    for _ in range(steps):
        dist = next_token_distribution(current)
        if not dist:
            break
        words, weights = zip(*dist.items())
        current = random.choices(words, weights=weights, k=1)[0]
        output.append(current)
    return ' '.join(output)

for _ in range(5):
    print(generate('language'))

## 8. Tiny model vs modern LLM
| Tiny lab model | Modern LLM |
|---|---|
| Mainly one previous token | Long context window |
| Counts transitions | Learns neural-network parameters |
| No embeddings | High-dimensional embeddings |
| No attention | Multi-head self-attention |
| Tiny corpus | Large-scale training data |
| Simple probabilities | Deep Transformer computation |

The shared principle is **context → next-token probabilities → choose token → append → repeat**.

## 9. Exercises
1. Add more sentences to the corpus and retrain.
2. Inspect distributions after `artificial`, `machine`, and `models`.
3. Change the logits and temperatures.
4. Explain: **logit**, **softmax**, **temperature**, **next-token prediction**.
5. Why can plausible next-token prediction still produce a factual error?

## Takeaway
A real LLM adds embeddings, positional information, self-attention, many Transformer blocks, large-scale pretraining and post-training. This notebook isolates the generation logic so you can see it working.